# FireChronicle — historical Aqua MODIS processing

**Region:** Pará, Brazil (`BR-PA`)  
**Processing period:** 2002–2011  
**Sensor:** Aqua MODIS only  
**Spatial reference:** EPSG:6933  
**Grid:** 568 fixed 50 km equal-area blocks

This notebook prepares the historical Aqua MODIS dataset for Pará from the annual NASA FIRMS Brazil country CSV files. Terra observations are excluded. Each year is processed separately using the IBGE 2024 Pará boundary and the project's canonical 50 km grid.

The historical workflow is separate from the MODIS–VIIRS overlap pipeline. It does not create VIIRS records, historical VIIRS estimates, or model predictions.

## Processing notes

- Aqua observations begin in July 2002. January–June 2002 is marked `not_observable_pre_aqua`.
- Detection counts are counts of satellite detections, not counts of individual fire events.
- Only positive Aqua detections are recorded in the sparse source table.
- Missing detections remain NULL because archive completeness has not been independently verified.
- Each annual output includes Parquet files, a ZIP package, and a JSON audit report.

## Inputs

- Annual source: `modis_YEAR_Brazil.csv`
- Boundary: `PA_UF_2024.zip` (IBGE 2024, Pará / `CD_UF = 15`)

The notebook downloads missing annual MODIS source files from the [NASA FIRMS country archive](https://firms.modaps.eosdis.nasa.gov/country/) and stores them in the Google Drive project folder.

## Outputs

Each year is written to its own folder under `Historical_MODIS_Outputs`. Existing annual folders are never overwritten. If an existing folder has a successful audit report, the notebook leaves it unchanged and moves on.

> Raw source CSVs and generated Parquet/ZIP outputs are kept in Google Drive, not committed to Git. The notebook itself contains the reproducible processing workflow.

[Open in Google Colab](https://colab.research.google.com/github/inan001/firechronicle/blob/main/notebooks/firechronicle_historical_aqua_modis_2002_2011.ipynb)

## 1. Install packages and connect Google Drive

Run this in Google Colab. The annual source files and generated outputs are stored in the `FireChronicle` folder in your Drive.

In [ ]:
%pip -q install pandas numpy pyarrow geopandas "shapely>=2" pyproj requests

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/FireChronicle")
INPUT = ROOT / "Input" / "NASA_FIRMS_Brazil"
OUTPUTS = ROOT / "Historical_MODIS_Outputs"

INPUT.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

print("Raw source folder:", INPUT)
print("Annual output folder:", OUTPUTS)

## 2. Select the years and locate the boundary

The default selection covers 2002–2011. You can process a smaller subset by editing `YEARS`.

The boundary ZIP should be placed in `FireChronicle/Input/`. If it is already in the NASA source folder, the notebook will use that copy instead.

In [ ]:
YEARS = list(range(2002, 2012))
DOWNLOAD_MISSING_SOURCE_FILES = True

BOUNDARY = ROOT / "Input" / "PA_UF_2024.zip"
if not BOUNDARY.exists() and (INPUT / "PA_UF_2024.zip").exists():
    BOUNDARY = INPUT / "PA_UF_2024.zip"

if not BOUNDARY.is_file():
    raise FileNotFoundError(
        f"Place the approved PA_UF_2024.zip boundary file at {ROOT / 'Input'}"
    )

assert len(set(YEARS)) == len(YEARS), "YEARS contains duplicates."
assert all(2002 <= year <= 2011 for year in YEARS), (
    "The historical Aqua MODIS pipeline accepts years 2002–2011."
)

print("Selected years:", YEARS)
print("Boundary file:", BOUNDARY)

## 3. Download any missing NASA source files

The downloader preserves files that are already present. A new download is written to a temporary `.part` file and moved into place only after the response passes basic CSV checks.

Set `DOWNLOAD_MISSING_SOURCE_FILES = False` if you want the notebook to stop when a source file is missing instead of downloading it.

In [ ]:
import time
import requests

def fetch_source(year):
    destination = INPUT / f"modis_{year}_Brazil.csv"
    url = (
        "https://firms.modaps.eosdis.nasa.gov/data/country/modis/"
        f"{year}/modis_{year}_Brazil.csv"
    )

    if destination.exists():
        if destination.stat().st_size <= 100:
            raise ValueError(f"Existing source file is unexpectedly small: {destination}")
        print("Using existing source:", destination.name)
        return destination

    if not DOWNLOAD_MISSING_SOURCE_FILES:
        raise FileNotFoundError(
            f"Missing {destination.name}. Download it from NASA FIRMS into {INPUT}."
        )

    temporary_file = destination.with_suffix(".csv.part")

    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with temporary_file.open("wb") as output:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            output.write(chunk)

            with temporary_file.open("rb") as source:
                header = source.read(2048).lower()

            required_header_items = (b",", b"acq_date", b"satellite")
            if not all(item in header for item in required_header_items) or b"<html" in header:
                raise ValueError(f"The response does not look like a FIRMS CSV: {url}")

            if temporary_file.stat().st_size <= 100:
                raise ValueError(f"Downloaded file is unexpectedly small: {url}")

            temporary_file.replace(destination)
            print(
                "Downloaded:",
                destination.name,
                f"({destination.stat().st_size / 1_000_000:.2f} MB)"
            )
            return destination

        except Exception as exc:
            if temporary_file.exists():
                temporary_file.unlink()

            if attempt == 2:
                raise RuntimeError(
                    f"Could not download {url}. You can download the annual CSV "
                    f"manually and place it in {INPUT}. Details: {exc}"
                ) from exc

            time.sleep(2 ** attempt)

for year in YEARS:
    audit_path = (
        OUTPUTS
        / f"historical_modis_{year}"
        / f"audit_historical_modis_{year}.json"
    )

    if audit_path.is_file():
        existing_audit = json.loads(audit_path.read_text(encoding="utf-8"))
        if existing_audit.get("validation", {}).get("passed") is True:
            print(f"Source download not needed for {year}: validated output already exists.")
            continue

    fetch_source(year)

## 4. Historical processing functions

This cell contains the Aqua-only processing implementation from the supplied notebook. It is presented as regular Python code so the processing steps can be inspected and reviewed.

The processing calculations, spatial assignment, schema, audit fields, and no-overwrite check are retained. The command-line wrapper has been removed from the notebook cell because the next section calls `process_year()` directly.

In [ ]:
"""FireChronicle historical Aqua MODIS-only, Pará, July 2002–December 2011.

No VIIRS fields, no inferred detection zeros, no historical model predictions.
Input annual FIRMS Brazil MODIS CSV: modis_YEAR_Brazil.csv
Boundary: exact approved IBGE 2024 PA_UF_2024.zip
"""
from __future__ import annotations
import hashlib
import json
import shutil
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely import intersects_xy
from shapely.geometry import box

REGION = 'BR-PA'
BLOCK_METRES = 50000
KEY = ['region_id','block_id','block_ix','block_iy','year','month','daynight']
COLS = KEY + ['modis_count','modis_frp_sum','coverage_status','processing_status',
               'modis_source_version','archive_completeness_verified','source_product']
COMP_COLS = ['region_id','year','month','daynight','sensor','processing_status',
             'coverage_status','archive_completeness_verified','source_product',
             'source_version','observation_evidence','notes']
BKEY = ['region_id','block_id','block_ix','block_iy']


def assert_that(condition, description):
    if not bool(condition):
        raise AssertionError(description)


def checksum(file: Path) -> str:
    h = hashlib.sha256()
    with open(file,'rb') as inp:
        for piece in iter(lambda: inp.read(1024*1024), b''):
            h.update(piece)
    return h.hexdigest()


def load_boundary(file: Path):
    assert_that(file.exists(),f'Boundary missing: {file}')
    with zipfile.ZipFile(file) as z:
        names = {Path(n).name for n in z.namelist()}
        assert_that(all(f'PA_UF_2024.{ext}' in names for ext in ('shp','shx','dbf','prj')),
                    'Expected approved PA_UF_2024 shapefile members')
        with tempfile.TemporaryDirectory() as td:
            td = Path(td)
            for n in z.namelist():
                if Path(n).name.startswith('PA_UF_2024.') and Path(n).suffix.lower() in {'.shp','.shx','.dbf','.prj','.cpg'}:
                    (td/Path(n).name).write_bytes(z.read(n))
            frame = gpd.read_file(td/'PA_UF_2024.shp')
    assert_that(len(frame)==1 and str(frame['CD_UF'].iloc[0])=='15','Expected exactly Pará, CD_UF=15')
    assert_that(frame.crs is not None and frame.crs.to_epsg()==4674,'IBGE CRS expected EPSG:4674')
    assert_that(frame.geometry.is_valid.all(),'Pará geometry invalid')
    return frame.to_crs(4326).geometry.iloc[0], frame.to_crs(6933).geometry.iloc[0]


def canonical_blocks(para_6933):
    x0,y0,x1,y1 = para_6933.bounds
    rows=[]
    for ix in range(int(np.floor(x0/BLOCK_METRES)),int(np.floor(x1/BLOCK_METRES))+1):
        for iy in range(int(np.floor(y0/BLOCK_METRES)),int(np.floor(y1/BLOCK_METRES))+1):
            sq=box(ix*BLOCK_METRES,iy*BLOCK_METRES,(ix+1)*BLOCK_METRES,(iy+1)*BLOCK_METRES)
            a=sq.intersection(para_6933).area
            if a>0:
                rows.append((REGION,f'{ix}_{iy}',ix,iy,a/1e6))
    g=pd.DataFrame(rows,columns=BKEY+['para_area_km2']).sort_values(['block_ix','block_iy']).reset_index(drop=True)
    assert_that(len(g)==568, f'Frozen grid expected 568 blocks, got {len(g)}')
    assert_that(g.block_id.is_unique, 'Duplicate canonical block IDs')
    assert_that(abs(g.para_area_km2.sum()-para_6933.area/1e6)<.01,'Grid clipping area reconciliation failed')
    return g


def collect_positive_aqua(csvfile: Path, year: int, para_4326, grid: pd.DataFrame):
    assert_that(csvfile.is_file(),f'MODIS annual source missing: {csvfile}')
    trans=Transformer.from_crs('EPSG:4326','EPSG:6933',always_xy=True)
    n_all=0; n_aqua=0; n_terra=0; n_pa=0
    found=[]; versions=set(); first=None; last=None; seen_years=set(); seen_satellites=set()
    required={'latitude','longitude','acq_date','satellite','daynight','version','frp'}
    for chunk in pd.read_csv(csvfile,chunksize=160000,low_memory=False):
        n_all+=len(chunk)
        assert_that(required.issubset(chunk.columns),f'Source missing {sorted(required-set(chunk.columns))}')
        dates=pd.to_datetime(chunk['acq_date'],errors='coerce')
        assert_that(dates.notna().all(),'Invalid acquisition date(s)')
        assert_that(dates.dt.year.eq(year).all(),f'Source includes dates outside {year}')
        seen_years.update(dates.dt.year.unique().tolist())
        sat=chunk['satellite'].astype(str).str.strip()
        seen_satellites.update(sat.unique().tolist())
        assert_that(set(sat.unique()).issubset({'Aqua','Terra'}),f'Unrecognized MODIS satellite value(s): {set(sat.unique())}')
        n_terra+=int(sat.eq('Terra').sum())
        d=chunk.loc[sat.eq('Aqua')].copy()
        n_aqua+=len(d)
        if not len(d):continue
        assert_that(d.daynight.isin(['D','N']).all(),'Day/night values must be D or N')
        assert_that(d.latitude.notna().all() and d.longitude.notna().all(),'Missing coordinate(s)')
        assert_that(d.latitude.between(-90,90).all() and d.longitude.between(-180,180).all(),'Invalid coordinates')
        d['acq_date']=pd.to_datetime(d['acq_date'],errors='raise')
        if year==2002:
            assert_that(d.acq_date.ge(pd.Timestamp('2002-07-01')).all(),
                        'Aqua observations before July 2002: investigate source')
        in_para=intersects_xy(para_4326,d.longitude.to_numpy(dtype='float64'),d.latitude.to_numpy(dtype='float64'))
        d=d.loc[in_para].copy()
        n_pa+=len(d)
        if not len(d):continue
        xx,yy=trans.transform(d.longitude.to_numpy(dtype='float64'),d.latitude.to_numpy(dtype='float64'))
        ix=np.floor(xx/BLOCK_METRES).astype('int64'); iy=np.floor(yy/BLOCK_METRES).astype('int64')
        d['block_ix']=ix;d['block_iy']=iy
        d['block_id']=pd.Series(ix.astype(str),index=d.index)+'_'+pd.Series(iy.astype(str),index=d.index)
        assert_that(set(d.block_id).issubset(set(grid.block_id)),'Clipped detection outside frozen canonical grid')
        d['region_id']=REGION;d['year']=year;d['month']=d.acq_date.dt.month
        d['frp']=pd.to_numeric(d.frp,errors='coerce')
        d['version']=d.version.astype('string')
        versions.update(d.version.dropna().astype(str).unique())
        minimum=str(d.acq_date.min().date());maximum=str(d.acq_date.max().date())
        first=min(first,minimum) if first else minimum
        last=max(last,maximum) if last else maximum
        found.append(d[KEY+['frp','version']])
    assert_that(n_all>0,'Empty NASA annual MODIS file')
    assert_that(n_all==n_aqua+n_terra,'Aqua/Terra filtering mismatch')
    # This specifically does not assert the yearly presence of all months or day-night periods.
    if found:
        points=pd.concat(found,ignore_index=True)
        aggregate=points.groupby(KEY,as_index=False).agg(
            modis_count=('frp','size'),
            modis_frp_sum=('frp',lambda s:s.sum(min_count=1)),
            modis_source_version=('version',lambda s:'|'.join(sorted(set(s.dropna().astype(str))))))
        aggregate.modis_count=aggregate.modis_count.astype('Int64')
    else:
        aggregate=pd.DataFrame(columns=KEY+['modis_count','modis_frp_sum','modis_source_version'])
        aggregate['modis_count']=aggregate.modis_count.astype('Int64')
    report={'source_rows':n_all,'aqua_brazil_rows':n_aqua,'terra_excluded_rows':n_terra,
            'aqua_para_rows':n_pa,'aqua_excluded_outside_para':n_aqua-n_pa,
            'source_satellites':sorted(seen_satellites),'source_years':sorted(seen_years),
            'para_aqua_first_detection_date':first,'para_aqua_last_detection_date':last,
            'para_aqua_source_versions':sorted(versions),
            'positive_aggregate_keys':len(aggregate)}
    assert_that(int(aggregate.modis_count.sum())==n_pa,'Positive detections do not reconcile to source')
    return aggregate,report


def process_year(year: int, input_dir: Path, boundary: Path, output_root: Path, reference_grid: Path|None=None):
    assert_that(2002<=year<=2011,'Historical pipeline only accepts 2002–2011')
    output_dir=output_root/f'historical_modis_{year}'
    assert_that(not output_dir.exists(),f'Output exists: {output_dir}; no overwriting')
    csvfile=input_dir/f'modis_{year}_Brazil.csv'
    polygon,para=load_boundary(boundary)
    grid=canonical_blocks(para)
    if reference_grid is not None:
        ref=pd.read_parquet(reference_grid)
        assert_that(set(BKEY).issubset(ref.columns),'Reference grid does not contain frozen block columns')
        a=grid[BKEY].sort_values(BKEY).reset_index(drop=True)
        b=ref[BKEY].sort_values(BKEY).reset_index(drop=True)
        assert_that(a.equals(b),'Grid does not match reference canonical block IDs and indices')
    positive,source=collect_positive_aqua(csvfile,year,polygon,grid)
    periods=pd.DataFrame([(year,m,dn) for m in range(1,13) for dn in ('D','N')],columns=['year','month','daynight'])
    panel=grid[BKEY].merge(periods,how='cross').merge(positive,on=KEY,how='left',validate='one_to_one',indicator=True)
    assert_that(int(panel['_merge'].eq('both').sum())==len(positive),'Positive detections lost on panel join')
    panel=panel.drop(columns='_merge')
    panel['modis_count']=panel.modis_count.astype('Int64')
    pre=panel.year.eq(2002)&panel.month.lt(7)
    panel['coverage_status']=np.where(pre,'not_observable_pre_aqua','unknown')
    panel['processing_status']=np.where(pre,'not_applicable_pre_aqua','preliminary')
    panel['archive_completeness_verified']=False
    panel['source_product']='NASA FIRMS MODIS annual Brazil CSV; Aqua-only'
    assert_that(panel.loc[pre,'modis_count'].isna().all(),'No Aqua counts allowed Jan–June 2002')
    assert_that(int(panel.modis_count.sum())==source['aqua_para_rows'],'Panel changed Aqua counts')
    assert_that(panel.modis_count.dropna().gt(0).all(),'No inferred zeros permitted')
    assert_that(panel.coverage_status.isin(['unknown','not_observable_pre_aqua']).all(),'Unknown coverage modified')
    assert_that(len(panel)==13632 and panel.groupby('block_id').size().eq(24).all(),
                'Frozen 568 blocks × 24 year periods not preserved')
    assert_that(not panel.duplicated(KEY).any(),'Duplicate historical panel keys')
    panel=panel[COLS].sort_values(KEY).reset_index(drop=True)
    comp=periods.copy();comp['region_id']=REGION;comp['sensor']='AQUA_MODIS'
    before=comp.year.eq(2002)&comp.month.lt(7)
    comp['processing_status']=np.where(before,'not_applicable_pre_aqua','preliminary')
    comp['coverage_status']=np.where(before,'not_observable_pre_aqua','unknown')
    comp['archive_completeness_verified']=False
    comp['source_product']='NASA FIRMS MODIS annual Brazil CSV; Aqua-only'
    comp['source_version']='|'.join(source['para_aqua_source_versions']) or pd.NA
    comp['observation_evidence']=np.where(before,
        'Aqua historical period begins July 2002; Jan-Jun not observed',
        'Positive-detection archive processed; independent sensor coverage not verified')
    comp['notes']=np.where(before,
        'Not a fire-free period; no Aqua observations to infer',
        'No positive detections => NULL, not verified zero or clear sky')
    comp=comp[COMP_COLS].sort_values(['year','month','daynight']).reset_index(drop=True)
    output_root.mkdir(parents=True,exist_ok=True)
    stage=Path(tempfile.mkdtemp(prefix=f'.fc_modis_{year}_staging_',dir=output_root))
    try:
        outputs={
          f'historical_modis_block_month_{year}.parquet':panel,
          f'historical_modis_positive_only_{year}.parquet':positive,
          f'historical_modis_completeness_{year}.parquet':comp,
          'para_50km_grid_canonical.parquet':grid[BKEY]
        }
        for name,data in outputs.items():
            data.to_parquet(stage/name,index=False)
        report={
          'pipeline':'FireChronicle historical Aqua MODIS-only',
          'year':year,'region_id':REGION,'created_utc':datetime.now(timezone.utc).isoformat(),
          'source':{'archive':'NASA FIRMS annual country Brazil standard archive','url':f'https://firms.modaps.eosdis.nasa.gov/data/country/modis/{year}/modis_{year}_Brazil.csv',
                    'sensor_selected':'Aqua','sensor_excluded':'Terra',**source},
          'inputs':{p.name:{'bytes':p.stat().st_size,'sha256':checksum(p)} for p in (csvfile,boundary)},
          'boundary_grid':{'source':'IBGE 2024 PA_UF_2024.zip','region_code':'15','boundary_crs':'EPSG:4674','point_coordinates_crs':'EPSG:4326',
                           'grid_crs':'EPSG:6933','block_size_metres':BLOCK_METRES,'origin_metres':[0,0],
                           'boundary_point_policy':'intersects (includes edges)','block_policy':'positive-area intersection',
                           'canonical_blocks':len(grid),'para_area_km2':float(grid.para_area_km2.sum()),
                           'reference_grid_checked':reference_grid is not None},
          'rows':{'full_panel':len(panel),'positive_keys':len(positive),'completeness_periods':len(comp)},
          'counts':{'aqua_para_positive_detections':int(panel.modis_count.sum()),
                    'null_counts':int(panel.modis_count.isna().sum()),
                    'explicit_zeros':int(panel.modis_count.eq(0).fillna(False).sum()),
                    '2002_pre_aqua_cells':int(pre.sum()),
                    'no_viirs_data_generated':True},
          'completeness':{'coverage_independently_verified':False,'no_detection_treated_as_null':True,
                          '2002_january_to_june':'not_observable_pre_aqua' if year==2002 else 'not_applicable',
                          'no_inferred_zeros':True,'no_viirs_input_or_output':True,'no_historical_model_predictions':True},
          'schema':{'full_panel':COLS,'period_completeness':COMP_COLS,'positive_only':KEY+['modis_count','modis_frp_sum','modis_source_version']},
          'validation':{'passed':True,'annual_year_checked':True,'positive_counts_reconciled':True,'grid_568':True,
                        'panel_rows_13632':True,'24_periods_per_block':True,'unique_keys':True,
                        'unverified_coverage_null':True,'pre_aqua_periods_explicit':True}
        }
        report['outputs']={p.name:{'bytes':p.stat().st_size,'sha256':checksum(p)} for p in sorted(stage.glob('*.parquet'))}
        audit_file=stage/f'audit_historical_modis_{year}.json'
        audit_file.write_text(json.dumps(report,indent=2,default=str),encoding='utf8')
        (stage/'README.txt').write_text(
            f'FireChronicle Aqua MODIS-only Pará historical annual archive, {year}.\n'
            'Frozen IBGE 2024 state polygon and 568-block 50km equal-area grid.\n'
            'Full panel is one row per block x month x D/N. Positive FIRMS detections only; unverified zero counts stay NULL.\n'
            'Jan-June 2002 is explicitly not observable for Aqua. No VIIRS or model predictions.\n'
            f'See audit_historical_modis_{year}.json for SHA-256 source and output provenance.\n',encoding='utf8')
        zipname=f'FireChronicle_Historical_MODIS_{year}.zip'
        with zipfile.ZipFile(stage/zipname,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
            for p in sorted(stage.iterdir()):
                if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
        stage.rename(output_dir)
        print(json.dumps({'year':year,'output':str(output_dir),'counts':report['counts'],'validation':'passed'},indent=2))
        return report
    except Exception:
        shutil.rmtree(stage,ignore_errors=True)
        raise


def main():
    parser=argparse.ArgumentParser(description='Historical Aqua MODIS-only Pará pipeline')
    parser.add_argument('--years',nargs='+',type=int,default=[2002,2003])
    parser.add_argument('--input-dir',type=Path,required=True)
    parser.add_argument('--boundary',type=Path,required=True)
    parser.add_argument('--output-root',type=Path,required=True)
    parser.add_argument('--reference-grid',type=Path,default=None)
    args=parser.parse_args()
    for year in args.years:
        process_year(year,args.input_dir,args.boundary,args.output_root,args.reference_grid)

if __name__=='__main__':
    main()


## 5. Process the selected years

Each year is handled independently. If an output folder already exists and its audit report records `validation.passed = true`, the notebook skips that year. If the folder exists but has no successful audit, the notebook stops for manual review rather than overwriting it.

This allows the full 2002–2011 range to remain selected while preserving annual outputs that were generated in earlier runs.

In [ ]:
import json
import shutil

LOCAL_CACHE = Path("/content/fc_historical_source_cache")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

local_boundary = LOCAL_CACHE / "PA_UF_2024.zip"
shutil.copy2(BOUNDARY, local_boundary)

processed_years = []
skipped_years = []

for year in YEARS:
    output_dir = OUTPUTS / f"historical_modis_{year}"
    audit_path = output_dir / f"audit_historical_modis_{year}.json"

    if output_dir.exists():
        if not audit_path.is_file():
            raise FileExistsError(
                f"{output_dir} already exists but has no audit report. "
                "Review the folder manually; it will not be overwritten."
            )

        existing_audit = json.loads(audit_path.read_text(encoding="utf-8"))
        recorded_outputs = existing_audit.get("outputs", {})
        outputs_match = bool(recorded_outputs) and all(
            (output_dir / name).is_file()
            and checksum(output_dir / name) == details.get("sha256")
            for name, details in recorded_outputs.items()
        )

        if (
            existing_audit.get("validation", {}).get("passed") is True
            and outputs_match
        ):
            print(f"Skipping {year}: audit passed and recorded Parquet checksums match.")
            skipped_years.append(year)
            continue

        raise RuntimeError(
            f"{year} has an existing output folder, but its audit or output "
            "checksums need review. The folder will not be overwritten."
        )

    source_file = INPUT / f"modis_{year}_Brazil.csv"
    if not source_file.is_file():
        raise FileNotFoundError(f"Required source file is missing: {source_file}")

    local_source = LOCAL_CACHE / source_file.name
    shutil.copy2(source_file, local_source)

    print(f"Processing historical year {year}...")
    process_year(
        year=year,
        input_dir=LOCAL_CACHE,
        boundary=local_boundary,
        output_root=OUTPUTS,
    )

    processed_years.append(year)
    local_source.unlink(missing_ok=True)

print("\nProcessing loop finished.")
print("Newly processed years:", processed_years)
print("Previously validated years skipped:", skipped_years)
print("Annual outputs:", OUTPUTS)

## 6. Review the annual audit reports

The audit report records the source and output checksums, the number of positive Aqua detections, NULL counts, grid dimensions, and validation flags.

This review cell reports the status of each selected year. It does not recalculate the datasets or assume that an output exists.

In [ ]:
import pandas as pd

audit_rows = []

for year in YEARS:
    output_dir = OUTPUTS / f"historical_modis_{year}"
    audit_path = output_dir / f"audit_historical_modis_{year}.json"

    if not audit_path.is_file():
        audit_rows.append({
            "year": year,
            "status": "audit not found",
            "output_files_match": False,
            "aqua_para_detections": None,
            "null_cells": None,
        })
        continue

    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    recorded_outputs = audit.get("outputs", {})
    outputs_match = bool(recorded_outputs) and all(
        (output_dir / name).is_file()
        and checksum(output_dir / name) == details.get("sha256")
        for name, details in recorded_outputs.items()
    )
    validation_passed = audit.get("validation", {}).get("passed") is True

    if validation_passed and outputs_match:
        status = "validated; checksums match"
    elif validation_passed:
        status = "audit passed; output check needs review"
    else:
        status = "review required"

    audit_rows.append({
        "year": year,
        "status": status,
        "output_files_match": outputs_match,
        "aqua_para_detections": audit.get("counts", {}).get(
            "aqua_para_positive_detections"
        ),
        "null_cells": audit.get("counts", {}).get("null_counts"),
    })

audit_summary = pd.DataFrame(audit_rows)
display(audit_summary)

## Output structure

Each successfully processed year has its own folder:

```text
Historical_MODIS_Outputs/
└── historical_modis_YEAR/
    ├── historical_modis_block_month_YEAR.parquet
    ├── historical_modis_positive_only_YEAR.parquet
    ├── historical_modis_completeness_YEAR.parquet
    ├── para_50km_grid_canonical.parquet
    ├── audit_historical_modis_YEAR.json
    ├── README.txt
    └── FireChronicle_Historical_MODIS_YEAR.zip
```

The full panel contains one record for each canonical block × month × day/night period. Unverified no-detection records remain NULL. January–June 2002 is explicitly marked as a period before Aqua observations began.

## Reproducibility and interpretation

- The audit JSON is the record of the exact source and output files used for each annual run.
- The notebook does not infer clear-sky conditions or fire-free periods from missing detections.
- No VIIRS data or historical model estimates are generated in this workflow.
- Saved outputs are not included in the GitHub notebook. Run the notebook in Colab to inspect or regenerate the annual audit reports.

Before publishing, confirm that each year you intend to report as complete has an audit file with `validation.passed = true`.